In [1]:
import pandas as pd
import numpy as np
import sys
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.ensemble import IsolationForest
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from mpl_toolkits.mplot3d import Axes3D # Nécessaire pour la 3D

# Configuration des chemins
ROOT_DIR = Path(__file__).resolve().parent.parent.parent
sys.path.insert(0, str(ROOT_DIR / "src"))

try:
    from config import DATA_PROCESSED, COL_PARTICIPANT, SIGNAL_COLS
except ImportError:
    DATA_PROCESSED = ROOT_DIR / "data" / "03_processed" / "dataset_final.csv"
    COL_PARTICIPANT = 'participant'
    SIGNAL_COLS = ["acc_x", "acc_y", "acc_z", "eda", "wrist_hr", "ibi", "temp", "breathing_rpm"]

# 1. Chargement
df_final = pd.read_csv(DATA_PROCESSED)

# 2. Profils par participant
profiles = df_final.groupby(COL_PARTICIPANT)[SIGNAL_COLS].agg(['mean', 'std']).fillna(0)
X = profiles.values
participant_ids = profiles.index

# 3. Normalisation et IsolationForest
X_scaled = StandardScaler().fit_transform(X)
clf = IsolationForest(random_state=42, contamination=0.15)
preds = clf.fit_predict(X_scaled) 

# 4. PCA à 3 composantes
pca = PCA(n_components=3)
X_pca = pca.fit_transform(X_scaled)
var_ratio = pca.explained_variance_ratio_

# 5. Graphique 3D
fig = plt.figure(figsize=(12, 9))
ax = fig.add_subplot(111, projection='3d')
sns.set_style("white")

colors = ['red' if p == -1 else 'royalblue' for p in preds]

# Tracer les points en 3D
scatter = ax.scatter(X_pca[:, 0], X_pca[:, 1], X_pca[:, 2], 
                     c=colors, s=200, edgecolors='k', alpha=0.7)

# Ajouter les labels des participants
for i, txt in enumerate(participant_ids):
    ax.text(X_pca[i, 0], X_pca[i, 1], X_pca[i, 2], f" P{txt:02d}", 
            fontsize=11, fontweight='bold')

# Configuration des axes
ax.set_title(f"Représentation 3D des Participants\nVariance totale expliquée : {sum(var_ratio):.1%}", fontsize=15)
ax.set_xlabel(f"CP1 ({var_ratio[0]:.1%})")
ax.set_ylabel(f"CP2 ({var_ratio[1]:.1%})")
ax.set_zlabel(f"CP3 ({var_ratio[2]:.1%})")

# Légende
from matplotlib.lines import Line2D
legend_elements = [Line2D([0], [0], marker='o', color='w', label='Normal', markerfacecolor='royalblue', markersize=12),
                   Line2D([0], [0], marker='o', color='w', label='Atypique', markerfacecolor='red', markersize=12)]
ax.legend(handles=legend_elements, loc='best', fontsize=12)

# Ajuster l'angle de vue initial pour une meilleure perspective
ax.view_init(elev=20, azim=45)

plt.tight_layout()
output_img = ROOT_DIR / "reports" / "plan_3d_participants.png"
plt.savefig(output_img)
print(f"✔ Le plan 3D a été sauvegardé dans : {output_img}")


NameError: name '__file__' is not defined

In [3]:
import pandas as pd

# Reading the Excel file
file_path = "/media/mohamedaziz-hadjayed/D/aziz_data/fatigue_detection/edge-ai-wearable-fatigue-detection/data/01_raw/preliminary_questionnaire.xlsx"
df = pd.read_excel(file_path)

# Renaming the columns
new_column_names = {
    "is reserved": "reserved",
    "… is generally trusting": "generally_trusting",
    "… tends to be lazy": "lazy",
    "… is relaxed, handles stress well": "relaxed_handless_stress",
    "… has few artistic interests": "few_artistic_interests",
    "… is outgoing, sociable": "sociable",
    "… tends to find fault with others": "criticize_others",
    "… does a thorough job": "thorough_job",
    "… gets nervous easily": "easily_nervous",
    "… has an active imagination": "active_magination",
    "Your general physical fitness is:": "phisical_fitness",
    "Your cardiorespiratory fitness (capacity to do exercise, for instance running, for a long time) is:": "cardiorespiratory_fitness",
    "Your muscular strength is:": "muscular_strength",
    "Your speed / agility is:": "agility_speed",
    "Your flexibility is:": "flexibility",
    "Do you have any existing/past cardiovascular health conditions (including but not limited to heart disease, high blood pressure, diabetes, asthama etc.?)?": "cardiovascular_health_conditions"
}

df = df.rename(columns=new_column_names)

# Writing the updated DataFrame to a CSV file
metadata_path = "/media/mohamedaziz-hadjayed/D/aziz_data/fatigue_detection/edge-ai-wearable-fatigue-detection/data/01_raw/metadata.csv"
df.to_csv(metadata_path, index=False)
